In [1]:
from pathlib import Path
import sys

ROOT = None
for p in Path("/kaggle/input").rglob("src"):
    if p.is_dir():
        ROOT = p.parent
        break

if ROOT is None:
    raise RuntimeError("Could not find repo root under /kaggle/input")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT


PosixPath('/kaggle/input/datasets/manishmodak/nemotron-reasoning-26')

In [2]:
from pathlib import Path

from src import load_config
from src.data import load_eval_examples
from src.eval import HeuristicPredictor, evaluate_examples
from src.eval.reporting import export_handoff_bundle
from src.eval.splits import split_examples
from src.prompts import get_prompt_variants
from src.solvers import ConservativeRouter

config = load_config(ROOT / "configs/experiments/baseline_eval.yaml")

train_path = None
for p in Path("/kaggle/input").rglob("train.csv"):
    if "nemotron" in str(p).lower():
        train_path = p
        break

if train_path is None:
    raise RuntimeError("Competition train.csv not found under /kaggle/input")

print("Using train file:", train_path)

all_examples = load_eval_examples(train_path)
train_examples, validation_examples = split_examples(
    all_examples,
    validation_ratio=config["split"]["validation_ratio"],
    seed=config["split"]["seed"],
)
selected_examples = validation_examples or all_examples
len(train_examples), len(selected_examples)


Using train file: /kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv


(4748, 4752)

In [3]:
from pathlib import Path

WORKDIR = Path("/kaggle/working/nemotron_baseline")
WORKDIR.mkdir(parents=True, exist_ok=True)

selected_prompt_names = set(config["prompts"])
prompt_variants = [
    variant
    for variant in get_prompt_variants()
    if variant.name in selected_prompt_names
]

router = ConservativeRouter(
    confidence_threshold=config["routing"]["confidence_threshold"]
)

selected_examples = selected_examples[:100]

result = evaluate_examples(
    selected_examples,
    prompt_variants=prompt_variants,
    predictor=HeuristicPredictor(),
    router=router if config["routing"]["enabled"] else None,
    run_name=config["run_name"],
    output_dir=WORKDIR / "artifacts/runs/baseline_eval",
)

handoff_bundle = export_handoff_bundle(
    result,
    WORKDIR / "artifacts/runs/kaggle_handoff",
    selected_variant=config["selected_prompt_variant"],
)

result.metrics


{'total_predictions': 300,
 'correct_predictions': 0,
 'overall_accuracy': 0.0,
 'accuracy_by_variant': {'baseline_direct': 0.0,
  'reasoned_boxed': 0.0,
  'self_check_boxed': 0.0},
 'routed_predictions': 0,
 'fallback_predictions': 300,
 'route_breakdown': {'llm_fallback': 300},
 'errors_by_family': {'bit_manipulation': 69, 'unknown_family': 231},
 'average_latency_s': 3.589579999546307e-05}

In [4]:
summary_path = result.artifact_paths["summary_md"]
print(summary_path.read_text(encoding="utf-8"))
print(f"Handoff bundle: {handoff_bundle}")


# Evaluation Summary: baseline_eval

- Total predictions: 300
- Overall accuracy: 0.000
- Routed predictions: 0
- Fallback predictions: 300

## Accuracy By Variant
- baseline_direct: 0.000
- reasoned_boxed: 0.000
- self_check_boxed: 0.000

## Routed-vs-Fallback Breakdown
- llm_fallback: 300

## Errors By Family
- bit_manipulation: 69
- unknown_family: 231

## Error Buckets
- parse_failure: 0
- symbolic_miss: 0
- llm_miss: 300

## Failure Sample
- 00066667
- 000b53cf
- 001b24c4
- 001c63cb
- 0047365c

Handoff bundle: /kaggle/working/nemotron_baseline/artifacts/runs/kaggle_handoff/handoff_bundle.json
